# WDI-style long-format data: LLM explanations for detected anomalies

This notebook targets **long-format** indicator exports in the spirit of the **World Development Indicators (WDI)**—one row per (indicator, country, year)—with your own detector columns. It is a **guided walkthrough** of the **explanation** stage: it assumes **detection is already done** (each row has the value *and* statistical detector outputs). The LLM does not re-run those detectors; it **reads the windowed context** around suspicious periods and returns **structured narratives** (external driver, data error, insufficient evidence, …) via `ai4data.anomaly.explanation`.

**Two stages, one pipeline**

| Stage | Who / what | This notebook |
|-------|----------------|---------------|
| **Detection** | Rules, z-scores, isolation forest, etc. | Uses your precomputed columns (e.g. `outlier_indicator_total`, `absZscore_zscore`). |
| **Explanation** | LLM + fixed prompts + JSON schema | Builds batch requests, runs provider batch APIs, parses results. |

**What you will do (in order)**

1. **Map** your file’s column names to the library’s **canonical** schema so every downstream step speaks one language.
2. **Optionally filter** rows by indicator code (e.g. poverty indicators `SI.*`) so cost and scope match your review question.
3. **Shortlist** `(indicator, geography)` series where anomalies are strong and detectors agree—this is where you spend LLM budget deliberately.
4. **Build JSONL** batch inputs for OpenAI, Gemini, and/or Anthropic; **optionally submit** jobs and **parse** JSONL outputs into tables you can audit.
5. **Optionally export** multi-provider results to JSON for the **anomaly explanation reviewer app** (Step 8).

### Prerequisites

- Install: `uv pip install -e ".[anomaly]"` from the repo root, or `uv pip install "ai4data[anomaly]"`.
- For remote batches: set `OPENAI_API_KEY`, `GEMINI_API_KEY` or `GOOGLE_API_KEY`, and `ANTHROPIC_API_KEY` as needed. Paths can use `~` if you wrap with `Path(...).expanduser()` (see configuration cells).

## End-to-end flow (data → prompts → answers)

The diagram is the **spine** of this notebook: each box matches a section below. Detection scores enter at the CSV; the LLM never sees raw column names from your export until they are renamed to the canonical schema.

```mermaid
flowchart LR
    wdi_long[WDI_style_long_format]
    map[Column_mapping]
    canon[Canonical_dataframe]
    filt[Indicator_filter]
    rank[Shortlist]
    jsonl[Batch_JSONL]
    llm[Provider_batch_API]
    out[parse_batch_output]
    review[Reviewer_JSON]
    wdi_long --> map --> canon --> filt --> rank --> jsonl --> llm --> out --> review
```

- **Mapping** — Aligns your export with the schema the prompts and parsers expect.
- **Filter** — Optional regex on indicator codes: defines *which themes* (e.g. poverty) get explanations, not only saves memory.
- **Shortlist** — Picks **which country–indicator series** to send to the LLM, ranked by severity and detector agreement.
- **JSONL → batch APIs** — Each line is one LLM request with the same system/user prompts; providers differ only in wire format.
- **parse_batch_output** — Turns provider JSONL into a table of explanations, classifications, and evidence fields you can review or export.
- **Reviewer JSON** (Step 8) — Optional `export_for_review_with_explainers` bundle for `apps.anomaly_review`.

## Canonical columns: what the library (and the LLM) rely on

The explanation code **does not** guess your column names. After mapping, each row must expose the fields below. They serve two roles: **ranking** which series to explain, and **building the prompt**: the model receives a **window of years** around flagged periods with values and imputation flags (see `extract_anomaly_contexts` in the library).

| Canonical | Role |
|-----------|------|
| `indicator_id`, `indicator_name` | What is measured (human-readable name helps the model ground text). |
| `geography_id`, `geography_name` | Where it is measured. |
| `period`, `value`, `is_imputed` | Time series points; **imputed** rows are excluded from the anomaly windows the LLM must explain. |
| `anomaly_score` | Magnitude (e.g. \|z-score\|)—used to **rank** series on the shortlist. |
| `outlier_count` | How many detectors flagged the row—**agreement** filters noise before LLM spend. |

Your export may use names like `outlier_indicator_total`; the mapping step connects that to `outlier_count`. Setting a **minimum** `outlier_count` (e.g. ≥ 3) is a policy choice: you only ask the LLM to narrate cases where several methods agree something is unusual.

In [13]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from ai4data.anomaly.explanation import (
    adapter_from_config,
    build_batch_file,
    export_for_review_with_explainers,
    load_csv_filtered,
    parse_batch_output,
    run_batch,
    suggest_column_mapping,
    # suggest_column_mapping_with_llm,
)


## Configuration: paths, scope, and environment

**CSV path** — Set `ANOMALY_CSV_PATH` or edit `CSV_PATH` below. Use `Path(...).expanduser().resolve()` so `~` in paths resolves to your home directory (raw `~` in a `Path` string is **not** expanded and can break `exists()`).

**Indicator filter** — `INDICATOR_ID_PATTERN` is a **regex** applied to the **source** indicator column *before* renaming (pandas `str.match`). Example: `r"^SI\."` limits runs to World Bank poverty / shared-prosperity style codes. Use `None` to load all indicators.

**Chunked reads** — For very large files, set `CHUNKSIZE` (e.g. `500_000`) so rows are filtered per chunk and memory stays bounded.

**Directories** — `ANOMALY_LLM_INPUT_DIR` / `ANOMALY_LLM_OUTPUT_DIR` override where batch JSONL inputs and provider outputs are written (default: next to your CSV under `llm-input` / `llm-output`).

**Batch toggle** — `RUN_ANOMALY_BATCH` (default `0`): set to `1` when you are ready to call provider APIs. Building JSONL files does **not** require API keys.

In [2]:
os.environ["ANOMALY_CSV_PATH"] = "~/WBG/ai4data/data/anomaly/wdi_data_anomalies_2026-04-07/wdi_data_anomalies_2026-04-07.CSV"

In [3]:
# --- User configuration ---
CSV_PATH = Path(os.environ["ANOMALY_CSV_PATH"]).expanduser().resolve()

INDICATOR_ID_PATTERN = r"^SI\."  # e.g. poverty / shared prosperity; set to None to load all indicators
CHUNKSIZE = None  # e.g. 500_000 for chunked reads

DATA_DIR = CSV_PATH.parent if CSV_PATH.name else Path(".")
INPUT_DIR = Path(os.environ.get("ANOMALY_LLM_INPUT_DIR", DATA_DIR / "llm-input"))
OUTPUT_DIR = Path(os.environ.get("ANOMALY_LLM_OUTPUT_DIR", DATA_DIR / "llm-output"))
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_OUTLIER_COUNT = 3
MAX_SERIES_FOR_DEMO = 1000
RUN_BATCH = bool(int(os.environ.get("RUN_ANOMALY_BATCH", "0")))

## Step 1 — Inspect headers and propose a column map

You are connecting **your** CSV to the **canonical** contract described above. Reading only the header with `pd.read_csv(..., nrows=0)` uses the **same quoting and encoding** as the full load (unlike a naive shell `head`, which can disagree in edge cases). For a quick shell check: `head -n1 your_file.csv`.

`suggest_column_mapping` guesses common names (e.g. `Country.Code` → `geography_id`). **Always review** the result: files with both `Zscore` and `absZscore_zscore`, for example, require you to choose which column drives `anomaly_score`. Optionally call `suggest_column_mapping_with_llm` once (OpenAI or Gemini) for a second opinion, then merge with your judgment.

In [4]:
if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Set CSV_PATH to your CSV. Current value: {CSV_PATH.resolve()}"
    )

header_columns = list(pd.read_csv(CSV_PATH, nrows=0).columns)
print("Columns:", header_columns)

heuristic_mapping = suggest_column_mapping(header_columns)
print("Heuristic mapping (canonical -> source):", heuristic_mapping)

Columns: ['Country.Code', 'Country.Name', 'Indicator.Code', 'Year', 'Value', 'Zscore', 'Indicator.Name', 'Imputed', 'outlier_indicator_tsoutlier', 'outlier_score_isotree', 'outlier_indicator_isotree', 'outlier_indicator_capa', 'capa_strength_capa', 'type_capa', 'outlier_score_outliertree', 'outlier_indicator_outliertree', 'absZscore_zscore', 'rankZscore_zscore', 'outlier_indicator_zscore', 'outlier_indicator_hampel', 'hampel_replacements_hampel', 'outlier_indicator_total']
Heuristic mapping (canonical -> source): {'geography_id': 'Country.Code', 'geography_name': 'Country.Name', 'indicator_id': 'Indicator.Code', 'period': 'Year', 'value': 'Value', 'anomaly_score': 'Zscore', 'indicator_name': 'Indicator.Name', 'is_imputed': 'Imputed', 'outlier_count': 'outlier_indicator_total'}


### Step 2 — Lock in the mapping (edit this cell)

Every **required** canonical key must point to exactly one source column. The cell below starts from heuristics then applies explicit overrides suitable for typical WDI-style exports (`Indicator.Code`, `absZscore_zscore`, `outlier_indicator_total`, …).

Uncomment the LLM helper if you want a one-shot suggestion from OpenAI or Gemini (one API call); your edits below should win over any automated guess.

In [5]:
# Start from heuristics, then override manually if needed
COLUMN_MAPPING = {
    **heuristic_mapping,
    "indicator_id": "Indicator.Code",
    "indicator_name": "Indicator.Name",
    "geography_id": "Country.Code",
    "geography_name": "Country.Name",
    "period": "Year",
    "value": "Value",
    "is_imputed": "Imputed",
    "anomaly_score": "absZscore_zscore",
    "outlier_count": "outlier_indicator_total",
}

# Optional: one-shot LLM suggestion (requires API key for chosen provider)
# llm_map = suggest_column_mapping_with_llm(header_columns, provider="openai")
# COLUMN_MAPPING = {**llm_map, **COLUMN_MAPPING}  # your edits win

adapter = adapter_from_config(COLUMN_MAPPING)
COLUMN_MAPPING

{'geography_id': 'Country.Code',
 'geography_name': 'Country.Name',
 'indicator_id': 'Indicator.Code',
 'period': 'Year',
 'value': 'Value',
 'anomaly_score': 'absZscore_zscore',
 'indicator_name': 'Indicator.Name',
 'is_imputed': 'Imputed',
 'outlier_count': 'outlier_indicator_total'}

## Step 3 — Load data and apply the indicator filter

`load_csv_filtered` applies `INDICATOR_ID_PATTERN` to the **source** indicator column, then renames columns to the canonical schema. Filtering here is both **practical** (smaller tables) and **substantive**: it defines the **population of anomalies** you will ask the LLM to explain—e.g. poverty only versus all sectors.

In [6]:
if CHUNKSIZE:
    canonical_df = load_csv_filtered(
        CSV_PATH,
        COLUMN_MAPPING,
        indicator_id_pattern=INDICATOR_ID_PATTERN,
        chunksize=CHUNKSIZE,
    )
else:
    canonical_df = load_csv_filtered(
        CSV_PATH,
        COLUMN_MAPPING,
        indicator_id_pattern=INDICATOR_ID_PATTERN,
    )

print(f"Rows: {len(canonical_df):,}; indicators: {canonical_df['indicator_id'].nunique()}")
canonical_df.head()

/Users/avsolatorio/WBG/ai4data/src/ai4data/anomaly/explanation/adapters.py:324: DtypeWarning: Columns (0: type_capa) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, **read_csv_kwargs)


Rows: 71,460; indicators: 19


,geography_id,geography_name,indicator_id,period,value,Zscore,indicator_name,is_imputed,outlier_indicator_tsoutlier,outlier_score_isotree,...,capa_strength_capa,type_capa,outlier_score_outliertree,outlier_indicator_outliertree,anomaly_score,rankZscore_zscore,outlier_indicator_zscore,outlier_indicator_hampel,hampel_replacements_hampel,outlier_count
160,CZE,Czechia,SI.POV.GAPS,2007,0.1,6.485242,Poverty gap at $3.00 a day (2021 PPP) (%),False,1.0,0.430300,...,6.485242,point,0,0,6.485242,162.0,1.0,1,-0.402130,4
166,SVN,Slovenia,SI.POV.GAPS,2005,0.1,6.484990,Poverty gap at $3.00 a day (2021 PPP) (%),False,1.0,0.419591,...,6.484990,point,0,0,6.484990,167.0,1.0,1,-0.402373,4
2077,KAZ,Kazakhstan,SI.DST.10TH.10,2005,29.7,5.967852,Income share held by highest 10%,False,1.0,0.395098,...,5.967852,point,0,0,5.967852,3011.0,1.0,1,-0.452321,4
2084,ARG,Argentina,SI.POV.LMIC,2002,25.4,5.965395,Poverty headcount ratio at $4.20 a day (2021 P...,False,1.0,0.404233,...,5.965395,point,0,0,5.965395,3023.0,1.0,1,-0.545043,4
2086,TUR,Turkiye,SI.POV.LMIC.GP,2003,6.1,5.965057,Poverty gap at $4.20 a day (2021 PPP) (%),False,1.0,0.395517,...,5.965057,point,0,0,5.965057,3025.0,1.0,1,-0.503603,4


## Step 4 — Shortlist series for LLM explanation

The LLM explains **one series at a time** in context: a **country** and **indicator** over a **window of years** around detected anomalies. The shortlist picks **which** `(indicator_id, geography_id)` pairs get that treatment.

Rows are filtered to non-imputed points with `outlier_count` at least `MIN_OUTLIER_COUNT`. Then each series is scored by the **maximum** `anomaly_score` among qualifying rows and sorted descending. `MAX_SERIES_FOR_DEMO` caps how many series enter the batch—raise it for production-sized runs once you trust the cost.

The **next code cell** builds `geo_map` / `ind_map` for human-readable labels and sets `source_df` indexed by `(indicator_id, geography_id)`—the shape `build_batch_file` expects when it pulls each series for context extraction.

In [7]:
filtered = canonical_df[
    (canonical_df["outlier_count"] >= MIN_OUTLIER_COUNT)
    & (~canonical_df["is_imputed"])
]

shortlist = pd.DataFrame()
if len(filtered):
    ranked = (
        filtered.groupby(["indicator_id", "geography_id"])["anomaly_score"]
        .max()
        .sort_values(ascending=False)
        .reset_index()
    )
    shortlist = ranked.head(MAX_SERIES_FOR_DEMO)

print(f"Shortlist: {len(shortlist)} series")
shortlist.head()

Shortlist: 714 series


,indicator_id,geography_id,anomaly_score
0,SI.POV.GAPS,CZE,6.485242
1,SI.POV.GAPS,SVN,6.484990
2,SI.POV.UMIC.GP,FIN,6.327177
3,SI.DST.04TH.20,XKX,6.139350
4,SI.DST.10TH.10,KAZ,5.967852


In [8]:
geo_map = canonical_df.drop_duplicates("geography_id").set_index("geography_id")[
    "geography_name"
].to_dict()
ind_map = canonical_df.drop_duplicates("indicator_id").set_index("indicator_id")[
    "indicator_name"
].to_dict()

source_df = canonical_df.set_index(["indicator_id", "geography_id"]).sort_index()

## Step 5 — Build batch JSONL (prompts, one request per context)

For each `(indicator_id, geography_id)` on the shortlist, the library **does not** send the whole CSV to the model. It:

1. Slices the canonical series for that pair.
2. Finds periods where `outlier_count` meets your threshold and builds **time windows** around them (see `extract_anomaly_contexts`).
3. Renders the **system** and **user** prompts from `ai4data.anomaly.explanation.prompts` and attaches a **JSON schema** so the model returns structured explanations (classifications, evidence, etc.).

Each resulting prompt becomes **one line** in a provider-specific JSONL file. The **logical content** is the same across OpenAI, Gemini, and Anthropic; only the **wire format** changes.

| Provider | Mechanism | Default model (override with `model_id` in code if you extend it) |
|----------|-----------|----------------------------------------------------------------------|
| **OpenAI** | Batch API, `/v1/responses` + JSON schema | `gpt-4.1-mini` |
| **Gemini** | GenAI batch file upload | `gemini-2.5-flash` |
| **Anthropic** | Message Batches + structured output | `claude-sonnet-4-6` (see `build_batch_file` defaults) |

Batch APIs favor **throughput and cost** over immediate responses—appropriate for overnight or large review runs, not chat-style debugging.

In [9]:
paths = {}
counts = {}
for provider in (
    "gemini",
    "anthropic",
    "openai",
):
    out, n = build_batch_file(
        INPUT_DIR / f"anomaly_batch_{provider}.jsonl",
        shortlist,
        source_df,
        geo_map,
        ind_map,
        provider=provider,
        min_outlier_count=MIN_OUTLIER_COUNT,
    )
    paths[provider] = out
    counts[provider] = n
    print(provider, "->", out, "requests:", n)

gemini -> /Users/avsolatorio/WBG/ai4data/data/anomaly/wdi_data_anomalies_2026-04-07/llm-input/anomaly_batch_gemini.jsonl requests: 726
anthropic -> /Users/avsolatorio/WBG/ai4data/data/anomaly/wdi_data_anomalies_2026-04-07/llm-input/anomaly_batch_anthropic.jsonl requests: 726
openai -> /Users/avsolatorio/WBG/ai4data/data/anomaly/wdi_data_anomalies_2026-04-07/llm-input/anomaly_batch_openai.jsonl requests: 726


## Step 6 — Submit batches and download results (optional)

`run_batch(provider, input_jsonl, output_jsonl)` is **synchronous**: it submits the job, **polls until completion** (default every 60 seconds), then **downloads** the result JSONL to `OUTPUT_DIR`. Each provider runs sequentially if you loop in a plain `for`—wall-clock time adds up. To run providers **in parallel**, execute separate `run_batch` calls in threads (e.g. `asyncio.to_thread`) so OpenAI, Gemini, and Anthropic batches progress concurrently.

**When to run** — Set `RUN_ANOMALY_BATCH=1` in the environment **or** set `RUN_BATCH = True` in the notebook after your keys are available. With `RUN_BATCH` false, you still produce valid JSONL under `INPUT_DIR` for manual upload in each vendor’s UI or CLI.

**Keys** — `OPENAI_API_KEY`, `GEMINI_API_KEY` or `GOOGLE_API_KEY`, `ANTHROPIC_API_KEY` must be set for the providers you actually call; missing keys raise clear errors from the client helpers.

In [10]:
RUN_BATCH = True

In [11]:
results = {}
if RUN_BATCH and len(shortlist):
    for provider in (
        # "anthropic",
        "gemini",
        # "openai",
    ):
        out_path = OUTPUT_DIR / f"anomaly_batch_{provider}_out.jsonl"
        results[provider] = run_batch(
            provider,
            paths[provider],
            out_path,
        )
        print(provider, "output:", results[provider])
else:
    print("Skipping remote batch (set RUN_ANOMALY_BATCH=1 to run).")

gemini output: /Users/avsolatorio/WBG/ai4data/data/anomaly/wdi_data_anomalies_2026-04-07/llm-output/anomaly_batch_gemini_out.jsonl


## Step 7 — Parse provider JSONL into reviewable tables

`parse_batch_output` reads the downloaded JSONL and produces a **flat table** of anomaly-level rows: classifications (e.g. external driver vs data error), free-text explanations, confidence, evidence strength, and optional evidence sources—aligned with the JSON schema used at generation time. Each row’s **`custom_id`** ties back to the indicator and geography encoded when the batch file was built, so you can join results to your shortlist and original series.

**`custom_id` map file** — When you build batches, `build_batch_file` writes `anomaly_batch_<provider>_custom_id_map.json` next to the **input** JSONL (under `INPUT_DIR`). Downloaded results live under `OUTPUT_DIR` as `anomaly_batch_<provider>_out.jsonl`. The parser looks for the map **next to the output file** by default; if your map stayed in `llm-input` only, pass **`custom_id_map_path`** pointing at that file (as below). Without the map, compact ids cannot be decoded and you get an **empty** DataFrame.

If an output file is missing (provider not run or job failed), the loop skips it and prints a short message. Use the heads of each DataFrame to spot-check whether explanations match your governance bar before scaling up `MAX_SERIES_FOR_DEMO`.

In [12]:
parsed = {}
for provider in (
    "openai",
    "gemini",
    # "anthropic",
):
    out_file = OUTPUT_DIR / f"anomaly_batch_{provider}_out.jsonl"
    if not out_file.exists():
        print("skip (no output file):", out_file)
        continue
    map_path = INPUT_DIR / f"anomaly_batch_{provider}_custom_id_map.json"
    parse_kw = {}
    if map_path.exists():
        parse_kw["custom_id_map_path"] = map_path
    parsed[provider] = parse_batch_output(
        out_file,
        provider,
        ind_map,
        geo_map,
        **parse_kw,
    )
    print(provider, "rows:", len(parsed[provider]))
    if parsed[provider].empty:
        print(
            "  (empty: ensure custom_id map exists — e.g.",
            map_path,
            "or next to the output JSONL)",
        )
    display(parsed[provider].head())

openai rows: 601


,custom_id,indicator_code,indicator,country_code,country,window,is_anomaly,classification,confidence,explanation,evidence_strength,evidence_source,source,window_str
0,a19911da0af0377e15f5a6e6f8d0b46ce2,SI.POV.GAPS,Poverty gap at $3.00 a day (2021 PPP) (%),CZE,Czechia,"[2007, 2007]",True,external_driver,0.9,The slight increase in poverty gap in 2007 may...,strong_direct,"[{'name': '2007-2008 Global Financial Crisis',...",llm_inferred,2007-2007
1,a1b45bb5af4c623875cdcf24858ca5ec1d,SI.POV.GAPS,Poverty gap at $3.00 a day (2021 PPP) (%),SVN,Slovenia,"[2005, 2005]",False,insufficient_data,0.7,The slight increase to 0.1% in 2005 is minimal...,no_evidence,[],llm_inferred,2005-2005
2,a14274e11ed9c44670302870b552abdee1,SI.POV.UMIC.GP,Poverty gap at $8.30 a day (2021 PPP) (%),FIN,Finland,"[2020, 2020]",True,external_driver,0.9,The drop to 0.0 in 2020 aligns with the onset ...,strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2020
3,a12438968bc2560e95c7752766fcf06735,SI.DST.04TH.20,Income share held by fourth 20%,XKX,Kosovo,"[2020, 2020]",True,external_driver,0.9,The sharp drop in income share held by the fou...,strong_direct,"[{'name': 'COVID-19 pandemic', 'date_range': '...",llm_inferred,2020-2020
4,a18b1f16f14e9947205d29b64f9789e276,SI.DST.10TH.10,Income share held by highest 10%,KAZ,Kazakhstan,"[2004, 2005]",True,external_driver,0.8,The sharp increase in income share held by the...,strong_direct,[{'name': 'Kazakhstan oil boom and economic gr...,llm_inferred,2004-2005


gemini rows: 890


,custom_id,indicator_code,indicator,country_code,country,window,is_anomaly,classification,confidence,explanation,evidence_strength,evidence_source,source,window_str
0,a15347839531cf42f5a01a278d15ffc89b,SI.POV.LMIC,Poverty headcount ratio at $4.20 a day (2021 P...,ISL,Iceland,"[2015, 2015]",True,data_error,0.9,The reported poverty headcount ratio of 0.0% i...,strong_direct,[],llm_inferred,2015-2015
1,a15347839531cf42f5a01a278d15ffc89b,SI.POV.LMIC,Poverty headcount ratio at $4.20 a day (2021 P...,ISL,Iceland,"[2017, 2017]",True,data_error,0.9,The reported poverty headcount ratio of 0.0% i...,strong_direct,[],llm_inferred,2017-2017
2,a15347839531cf42f5a01a278d15ffc89b,SI.POV.LMIC,Poverty headcount ratio at $4.20 a day (2021 P...,ISL,Iceland,"[2018, 2018]",False,insufficient_data,0.5,"No clear, well-documented event or data-relate...",no_evidence,[],llm_inferred,2018-2018
3,a18ad274132ddc84319c6a7be72be5f0df,SI.DST.04TH.20,Income share held by fourth 20%,MYS,Malaysia,"[1995, 1997]",False,insufficient_data,0.5,Insufficient non-imputed data points to identi...,no_evidence,[],llm_inferred,1995-1997
4,a1e6d1fbae1c4a8b97ac04e09b02976276,SI.POV.UMIC,Poverty headcount ratio at $8.30 a day (2021 P...,HIC,High income,"[1993, 1997]",True,external_driver,0.9,The elevated poverty headcount ratio in high-i...,strong_direct,[{'name': 'Early 1990s Recessions in High-Inco...,llm_inferred,1993-1997


## Step 8 — Export for the anomaly explanation reviewer app

This packages **several** parsed provider tables into one JSON file with **`explanation.explainers`** (one entry per model) so the reviewer UI can compare side by side. This uses `export_for_review_with_explainers`.

- Pass **`canonical_df`** as `timeseries_df` so each item includes a **`timeseries`** slice around the anomaly window (for charts). Omit it if you only need text.
- Output path defaults next to your data; override `REVIEW_JSON_PATH` or edit the cell below.
- **Run the app** (from repo root): `uv run python -m apps.anomaly_review path/to/anomaly_review.json` — then open http://localhost:8000 (see `apps/anomaly_review/README.md`).

For a **single** provider, use `export_for_review` instead.

In [14]:
# Multi-provider JSON for apps/anomaly_review (Option B)
REVIEW_JSON_PATH = OUTPUT_DIR / "anomaly_review_multimodel.json"

explainer_dfs = []
label_for = {
    "openai": "OpenAI",
    "gemini": "Gemini",
    "anthropic": "Anthropic",
}
for key, label in label_for.items():
    df = parsed.get(key)
    if df is not None and not df.empty:
        explainer_dfs.append((label, df))

if not explainer_dfs:
    print("No parsed DataFrames: run Step 7 first or enable at least one provider.")
else:
    review_path = export_for_review_with_explainers(
        explainer_dfs,
        timeseries_df=canonical_df,
        output_path=REVIEW_JSON_PATH,
        period_padding=5,
        run_arbiter=False,
    )
    print("Wrote:", review_path.resolve())
    print("Run: uv run python -m apps.anomaly_review", review_path)

Wrote: /Users/avsolatorio/WBG/ai4data/data/anomaly/wdi_data_anomalies_2026-04-07/llm-output/anomaly_review_multimodel.json
Run: uv run python -m apps.anomaly_review /Users/avsolatorio/WBG/ai4data/data/anomaly/wdi_data_anomalies_2026-04-07/llm-output/anomaly_review_multimodel.json
